# Cumulative MTD Export Pace — Kpler Flows prototype

Pulls **daily export flows** for a location + calendar month from the Kpler **Flows** endpoint,
builds the cumulative month-to-date (MTD) curve, and overlays:

- **Current month** (partial, dotted) — truncated at today
- **Last month** and **Last year** (same as your Excel chart)
- **N-year seasonal band**: median + interquartile (25–75%) + min–max range, computed
  per *day-of-month* across the prior N years of the *same calendar month*

### Design decisions worth knowing (these are easy to get wrong)
1. **Flows is the right endpoint.** It returns aggregated import/export flows for a zone or
   installation at daily granularity — exactly the time series this chart needs. (`Trades` gives
   you cargo-level rows; you'd have to aggregate yourself. Don't.)
2. **Pull volume, not rate, then cumulate.** We request the unit **`KB`** (thousand barrels =
   volume *that day*) and `cumsum()` it, so the y-axis reads "thousand barrels shipped MTD".
   Your Excel chart is labelled *kbd* but is really this cumulative curve — cumulating a *rate*
   (kbd) isn't a clean physical unit, so KB is the honest choice. Switch the unit in config if
   your desk insists on the kbd convention; the shape is identical.
3. **Forecast handling.** Flows defaults to `with_forecast=True` (Kpler's model fills in expected
   loadings). For a clean MTD *pace* comparison you usually want **realized only** for history and
   a deliberate choice for the live month. Both flags are exposed in config — see notes at the bottom.
4. **Day-of-month alignment.** We index each month by day-of-month (1..31) so years line up. For the
   *same* calendar month across years this is clean (only Feb leap-day varies; handled nan-aware).
   "Last month" is a *different* calendar month, so it's a rough pace overlay — fine for context,
   not a like-for-like.

**Run it now with `USE_MOCK = True` to see the chart, then flip to `False` and add credentials.**

In [52]:
import calendar
from datetime import date
import numpy as np
import pandas as pd
from dotenv import load_dotenv
from pathlib import Path

# ----------------------------- CONFIG -----------------------------
USE_MOCK     = False          # True = synthetic data, runs with zero setup. False = real Kpler.

LOCATION     = ["Yanbu", "Fujairah", "Oman"]  # str or list[str] — zone / installation names
PRODUCT      = "Diesel"                     # str or list[str] — Kpler product name(s)
MONTH        = 6                  # calendar month to analyse (1-12)
TARGET_YEAR  = 2026               # the 'current' year for this view
N_HISTORY    = 5                  # band over the prior N same-calendar-months
UNIT         = "kb"               # 'kb' (volume/day -> clean cumsum) or 'kbd'
AS_OF_DAY    = None                 # last day to draw for the 'current month' line.
                                  #   None -> use today if it's the live month, else full month.

# Kpler API params (only used when USE_MOCK = False)
WITH_FORECAST_LIVE = False    # include model forecast for the current/partial month
WITH_FORECAST_HIST = False   # history: realized picture is usually what you want
ONLY_REALIZED_HIST = True
WITH_INTRA_COUNTRY = True    # count intra-country trades as exports? depends on the location

# Credentials (env vars recommended; never hard-code in a shared notebook)
import os

load_dotenv(dotenv_path=Path('.env'))


KPLER_EMAIL    = os.getenv("KPLER_EMAIL")
KPLER_PASSWORD = os.getenv("KPLER_PASSWORD")


def as_filter_list(value):
    """Normalize LOCATION / PRODUCT: one name or a list for Flows API filters."""
    if isinstance(value, str):
        return [value]
    return list(value)


def filter_label(value, *, sep=", "):
    """Readable label for titles (single name unchanged; lists joined)."""
    items = as_filter_list(value)
    return items[0] if len(items) == 1 else sep.join(items)


LOCATIONS = as_filter_list(LOCATION)
PRODUCTS = as_filter_list(PRODUCT)
LOCATION_LABEL = filter_label(LOCATION)
PRODUCT_LABEL = filter_label(PRODUCT)
# ------------------------------------------------------------------

In [53]:
KPLER_EMAIL

'luiscarlos.gaitan@jainglobal.com'

## 1. Data layer — real Kpler Flows call, with a mock fallback

In [54]:
_flows_client = None
_flows_bulk = {}  # cache key -> raw Flows DataFrame (one API call per policy + filters)


def _get_flows_client():
    """Lazily build one Flows client (reused across calls)."""
    global _flows_client
    if _flows_client is None:
        from kpler.sdk import Platform
        from kpler.sdk.configuration import Configuration
        from kpler.sdk.resources.flows import Flows
        config = Configuration(Platform.Liquids, KPLER_EMAIL, KPLER_PASSWORD)
        _flows_client = Flows(config)
    return _flows_client


def _flows_bulk_start_date():
    """Earliest calendar day needed for history band, overlays, and prior-year month."""
    return date(TARGET_YEAR - N_HISTORY - 1, 1, 1)


def _flows_bulk_cache_key(live):
    """Cache key so bulk pull refreshes when zones, products, or API flags change."""
    return (
        live,
        tuple(LOCATIONS),
        tuple(PRODUCTS),
        UNIT,
        WITH_FORECAST_LIVE if live else WITH_FORECAST_HIST,
        False if live else ONLY_REALIZED_HIST,
        WITH_INTRA_COUNTRY,
    )


def _fetch_flows_bulk(*, live=False):
    """One Flows request for the full span; sliced per (year, month) downstream."""
    key = _flows_bulk_cache_key(live)
    if key in _flows_bulk:
        return _flows_bulk[key]
    from kpler.sdk import (FlowsDirection, FlowsSplit, FlowsPeriod,
                           FlowsMeasurementUnit)
    unit_enum = {"kb": FlowsMeasurementUnit.KB,
                 "kbd": FlowsMeasurementUnit.KBD}[UNIT]
    df = _get_flows_client().get(
        flow_direction=[FlowsDirection.Export],
        split=[FlowsSplit.Total],
        granularity=[FlowsPeriod.Daily],
        from_zones=LOCATIONS,
        products=PRODUCTS,
        unit=[unit_enum],
        start_date=_flows_bulk_start_date(),
        end_date=date.today(),
        with_forecast=WITH_FORECAST_LIVE if live else WITH_FORECAST_HIST,
        only_realized=False if live else ONLY_REALIZED_HIST,
        with_intra_country=WITH_INTRA_COUNTRY,
    )
    _flows_bulk[key] = df
    return df


def _flows_df_to_daily(df, year, month):
    """Extract one calendar month from a Flows bulk frame (Date + Total columns)."""
    dt = pd.to_datetime(df["Date"])
    if "Total" in df.columns:
        values = pd.to_numeric(df["Total"], errors="coerce")
    else:
        # split != Total: sum numeric flow columns (exclude Period End Date, etc.)
        values = df.select_dtypes("number").sum(axis=1)
    s = pd.Series(values.to_numpy(), index=dt, name="export")
    return s[(s.index.month == month) & (s.index.year == year)].sort_index()


def _kpler_export_daily(year, month, *, live=False):
    """Daily export series for one month, sliced from the bulk Flows pull."""
    df = _fetch_flows_bulk(live=live)
    return _flows_df_to_daily(df, year, month)


def _mock_export_daily(year, month, *, live=False, base=600.0, seed=0):
    # Tie mock noise to the active zone/product filters.
    filter_seed = hash((tuple(LOCATIONS), tuple(PRODUCTS))) % (2**31)
    rng = np.random.default_rng(seed + filter_seed + year * 100 + month)
    n = calendar.monthrange(year, month)[1]
    trend = 1.0 + 0.03 * (year - 2020)
    seasonal = 1.0 + 0.12 * np.sin((month - 1) / 12 * 2 * np.pi)
    mean_daily = base * trend * seasonal
    cargo = 380.0
    daily = rng.poisson(1.6, n) * cargo * rng.uniform(0.6, 1.4, n)
    daily = daily * (mean_daily / max(daily.mean(), 1e-9))
    idx = pd.date_range(date(year, month, 1), periods=n, freq="D")
    return pd.Series(daily, index=idx, name="export")


_cache = {}
def get_export_daily(year, month, *, live=False):
    """Cached daily export series for one (year, month)."""
    key = (year, month, live, tuple(LOCATIONS), tuple(PRODUCTS))
    if key not in _cache:
        fn = _mock_export_daily if USE_MOCK else _kpler_export_daily
        _cache[key] = fn(year, month, live=live)
    return _cache[key]

## 2. Transform — cumulative by day-of-month, and the history matrix

In [55]:
def to_dom_cumulative(daily, *, as_of=None):
    """Reindex to day-of-month (1..N) and cumulative-sum. as_of truncates the live month."""
    s = daily.copy()
    if as_of is not None:
        s = s[s.index.date <= as_of]
    cum = s.cumsum()
    cum.index = cum.index.day
    return cum


def prev_month(year, month):
    return (year - 1, 12) if month == 1 else (year, month - 1)


def build_history_matrix(month, years):
    """day_of_month (rows) x year (cols) cumulative-export matrix."""
    return pd.DataFrame(
        {y: to_dom_cumulative(get_export_daily(y, month)) for y in years}
    ).sort_index()


def seasonal_band(hist):
    out = pd.DataFrame(index=hist.index)
    out["median"] = hist.median(axis=1)
    out["p25"]    = hist.quantile(0.25, axis=1)
    out["p75"]    = hist.quantile(0.75, axis=1)
    out["min"]    = hist.min(axis=1)
    out["max"]    = hist.max(axis=1)
    return out

## 3. Assemble the series for this view

In [56]:
# resolve 'as of' for the current-month line
today = date.today()
is_live_month = (TARGET_YEAR == today.year and MONTH == today.month)
if AS_OF_DAY is not None:
    as_of = date(TARGET_YEAR, MONTH, AS_OF_DAY)
elif is_live_month:
    as_of = today
else:
    as_of = None   # full month

hist_years = list(range(TARGET_YEAR - N_HISTORY, TARGET_YEAR))   # prior N same-months
hist = build_history_matrix(MONTH, hist_years)
band = seasonal_band(hist)

current  = to_dom_cumulative(get_export_daily(TARGET_YEAR, MONTH, live=True), as_of=as_of)
last_yr  = to_dom_cumulative(get_export_daily(TARGET_YEAR - 1, MONTH))
py, pm   = prev_month(TARGET_YEAR, MONTH)
last_mo  = to_dom_cumulative(get_export_daily(py, pm))

band.tail()

,median,p25,p75,min,max
Date,,,,,
26,9119.55,7242.34,9276.48,4456.82,10914.82
27,9119.55,7242.34,9622.65,4506.83,10914.82
28,9380.08,7870.32,10338.93,5249.93,11975.36
29,9693.39,8772.40,10338.93,5249.93,12295.25
30,10370.70,9117.75,11287.29,6032.30,13070.90


## 4. Plot (plotly — this `go.Figure` drops straight into a Dash `dcc.Graph`)

In [57]:
import plotly.graph_objects as go

month_name = calendar.month_name[MONTH]
unit_label = "kb" if UNIT == "kb" else "kbd (cumulated)"
x = band.index

fig = go.Figure()
# min-max range band
fig.add_trace(go.Scatter(x=x, y=band["max"], line=dict(width=0), hoverinfo="skip",
                         showlegend=False))
fig.add_trace(go.Scatter(x=x, y=band["min"], fill="tonexty", line=dict(width=0),
                         fillcolor="rgba(158,202,225,0.30)",
                         name=f"{N_HISTORY}y range (min–max)", hoverinfo="skip"))
# interquartile band
fig.add_trace(go.Scatter(x=x, y=band["p75"], line=dict(width=0), hoverinfo="skip",
                         showlegend=False))
fig.add_trace(go.Scatter(x=x, y=band["p25"], fill="tonexty", line=dict(width=0),
                         fillcolor="rgba(107,174,214,0.45)",
                         name=f"{N_HISTORY}y IQR (25–75%)", hoverinfo="skip"))
# median + overlays
fig.add_trace(go.Scatter(x=x, y=band["median"], name=f"{N_HISTORY}y median",
                         line=dict(color="#2171b5", width=2)))
fig.add_trace(go.Scatter(x=last_mo.index, y=last_mo.values, name="Last month",
                         line=dict(color="#9e9ac8", width=2)))
fig.add_trace(go.Scatter(x=last_yr.index, y=last_yr.values, name="Last year",
                         line=dict(color="#74c476", width=2)))
fig.add_trace(go.Scatter(x=current.index, y=current.values, name="Current month",
                         line=dict(color="#e6550d", width=3, dash="dot")))

fig.update_layout(
    title=f"Cumulative {PRODUCT_LABEL} exports — {LOCATION_LABEL} — {month_name} {TARGET_YEAR}"
          + ("  (MOCK DATA)" if USE_MOCK else ""),
    xaxis_title="Day of month", yaxis_title=f"Cumulative {unit_label} (MTD)",
    xaxis=dict(range=[1, 31], dtick=2), template="plotly_white",
    legend=dict(orientation="h", y=-0.2), height=520,
)
fig.show()

## 5. Mapping to the real API & parameters to think about

The live call is `Flows.get(...)`. The parameters that change your numbers:

| Param | What it does | Suggested |
|---|---|---|
| `flow_direction` | `Export` / `Import` / `NetExport` | `Export` |
| `split` | how the total is broken out | `Total` (one curve). Use `DestinationCountries` etc. later for stacked views |
| `granularity` | `Daily` / `Weekly` / `Monthly` | `Daily` for this chart |
| `from_zones` / `products` | export origin zones and product filters | pass one name or a list; Kpler returns one combined `Total` series |
| `from_zones` vs `from_installations` | country/region/port **vs** a specific terminal | zone for country-level; installation for a single terminal (e.g. Ras Tanura) |
| `unit` | `kb` (volume) vs `kbd` (rate) | `kb`, then cumsum |
| `with_forecast` | include Kpler's predicted loadings | `False` for history; decide per-desk for the live month |
| `only_realized` | only completed trades | `True` for clean history |
| `with_intra_country` | count within-country moves | depends on location/product |

**Validate the zone/product names first.** Use the `Zones` and `Products` dimension endpoints
(`kpler.sdk.resources.zones.Zones`, `...products.Products`) to confirm exact strings — Flows is
strict about names. A quick `zones_client.get(...)` lookup avoids silent empty results.

**Response shape.** `Flows.get` returns a `Date` column and `Total` (when `split=Total`), not a
DatetimeIndex — slice on `Date`, not the row index.

**Bulk pull.** Section 1 fetches one date range (at most two calls if live-month `only_realized`
differs from history), then slices each calendar month in pandas. For the live month, realized data
lags ~a day or two behind today.

## 6. Path to a Dash app (when you're ready)

The structure here is deliberately Dash-shaped — lift sections 1–3 into a `data.py` module and
section 4 into a `figure.py`, then the app is thin:

```python
# app.py (sketch)
import dash
from dash import dcc, html, Input, Output
from data import build_view          # wraps sections 1-3 -> (current,last_mo,last_yr,band)
from figure import make_figure        # wraps section 4 -> go.Figure

app = dash.Dash(__name__)
app.layout = html.Div([
    dcc.Dropdown(id="location", options=[...], value="Saudi Arabia"),
    dcc.Dropdown(id="product",  options=[...], value="crude"),
    dcc.Dropdown(id="month",    options=[{"label": m, "value": i} for i, m in ...]),
    dcc.Graph(id="chart"),
])

@app.callback(Output("chart","figure"),
              Input("location","value"), Input("product","value"), Input("month","value"))
def update(location, product, month):
    view = build_view(location, product, month)   # cache this!
    return make_figure(view)
```

Two things that matter once it's an app:
- **Cache the Kpler calls.** History (prior years, realized) is static — cache it on disk
  (`functools.lru_cache`, `joblib.Memory`, or a Snowflake table) so dropdown changes don't
  re-hit the API. Only the live month needs refreshing. This is also where your Snowflake setup
  fits: land facility-day flows into a table and read from there.
- **Populate dropdowns from the `Zones`/`Products` dimension endpoints** so users can't pick
  invalid names.

A notebook is the right call for *now*; promote to Dash once the data layer and the few stats
choices (band percentiles, forecast policy) feel settled.

In [58]:
from kpler.sdk import Platform
from kpler.sdk.configuration import Configuration
from kpler.sdk.resources.flows import Flows

config = Configuration(Platform.Liquids, KPLER_EMAIL, KPLER_PASSWORD)

from datetime import date
from kpler.sdk.resources.flows import Flows
from kpler.sdk import FlowsDirection, FlowsSplit, FlowsPeriod, FlowsMeasurementUnit
flows_client = Flows(config)
flows_client.get(
    flow_direction=[FlowsDirection.Export],
    split=[FlowsSplit.OriginCountries],
    granularity=[FlowsPeriod.Daily],
    unit=[FlowsMeasurementUnit.T],
    start_date=date(2020,9,1),
    end_date=date(2020,10,1)
)

,Date,Saudi Arabia,Russian Federation,United States,United Arab Emirates,Iraq,Kuwait,Nigeria,Brazil,South Korea,...,Iceland,Haiti,Djibouti,Ukraine,Maldives,Kenya,Faroe Islands,American Samoa,Jersey,Period End Date
0,2020-09-01,1337934.21,636386.25,574187.94,841001.94,399524.13,457877.08,24237.74,124774.25,264769.18,...,0.00,4400.16,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-01
1,2020-09-02,958901.00,1057946.88,236515.66,652074.27,0.00,341756.27,749881.02,79240.48,65390.71,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-02
2,2020-09-03,532283.41,298858.00,892535.97,511625.21,854799.39,376110.10,302693.16,381104.26,0.00,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-03
3,2020-09-04,832319.81,759667.17,718690.58,267976.37,288216.71,258271.38,49758.94,46458.20,311365.29,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-04
4,2020-09-05,1044677.22,738409.50,770725.54,400030.81,423682.49,76125.00,125941.60,466515.17,244992.40,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-05
5,2020-09-06,1442576.81,998300.57,265866.33,337562.39,508708.32,207326.63,245257.20,221406.29,111709.76,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-06
6,2020-09-07,768620.68,741548.50,571420.83,280531.67,392340.60,510043.93,136662.41,65928.09,1406.53,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-07
7,2020-09-08,811387.29,951279.17,879195.36,382277.08,684521.38,511456.62,360994.82,470171.26,44928.72,...,5696.15,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-08
8,2020-09-09,1087583.24,553425.05,614260.97,1022116.31,315924.45,23067.85,384081.82,81744.37,129193.76,...,0.00,0.00,0.00,0.0,0.00,0.00,0.00,0.00,0.0,2020-09-09
9,2020-09-10,777694.46,849886.47,377071.79,538033.88,496473.45,321808.18,392159.34,176169.69,306680.47,...,0.00,0.00,0.00,3439.7,0.00,0.00,0.00,0.00,0.0,2020-09-10


In [59]:
from kpler.sdk import (FlowsDirection, FlowsSplit, FlowsPeriod,
                        FlowsMeasurementUnit)
unit_enum = {"kb": FlowsMeasurementUnit.KB,
                "kbd": FlowsMeasurementUnit.KBD}[UNIT]

live = False

In [60]:
flows_client.get(
        flow_direction=[FlowsDirection.Export],
        split=[FlowsSplit.Total],
        granularity=[FlowsPeriod.Daily],
        from_zones=LOCATIONS,
        products=PRODUCTS,
        unit=[unit_enum],
        start_date=date(2019, 1, 1),
        end_date=date.today(),
        with_forecast=WITH_FORECAST_LIVE if live else WITH_FORECAST_HIST,
        only_realized=False if live else ONLY_REALIZED_HIST,
        with_intra_country=WITH_INTRA_COUNTRY,
    )

,Date,Total,Period End Date
0,2019-01-01,0.00,2019-01-01
1,2019-01-02,0.00,2019-01-02
2,2019-01-03,0.00,2019-01-03
3,2019-01-04,0.00,2019-01-04
4,2019-01-05,866.64,2019-01-05
...,...,...,...
2707,2026-05-31,0.00,2026-05-31
2708,2026-06-01,97.25,2026-06-01
2709,2026-06-02,0.00,2026-06-02
2710,2026-06-03,743.17,2026-06-03
